In [2]:
import tifffile
import dask.array as da
import napari
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
from skimage import io

## Load image

In [3]:
from pathlib import Path
import zarr
import dask.array as da

def open_ngff_pyramids(store_path, label_name="seg"):
    """
    Return (img_levels, lab_levels, img_axes, lab_axes)
      - img_levels: [dask.array, ...] for image multiscale (0 is highest res)
      - lab_levels: [dask.array, ...] for labels multiscale (may be single-scale)
      - img_axes / lab_axes: lists of axis dicts from NGFF metadata
    """
    store_path = Path(store_path)
    root = zarr.open_group(str(store_path), mode="r")

    # --- image pyramid ---
    img_ms = root.attrs["multiscales"][0]           # tolerate v0.4/v0.5
    img_axes = img_ms.get("axes", [{"name":"c"},{"name":"y"},{"name":"x"}])
    img_paths = [d["path"] for d in img_ms["datasets"]]
    img_levels = [da.from_zarr(str(store_path / p)) for p in img_paths]

    # --- labels pyramid (optional) ---
    lab_levels, lab_axes = [], [{"name":"y","type":"space"},{"name":"x","type":"space"}]
    lab_group_path = store_path / "labels" / label_name
    if lab_group_path.exists():
        lab_group = zarr.open_group(str(lab_group_path), mode="r")
        lab_ms = lab_group.attrs["multiscales"][0]
        lab_axes = lab_ms.get("axes", lab_axes)
        lab_paths = [d["path"] for d in lab_ms["datasets"]]
        lab_levels = [da.from_zarr(str(lab_group_path / p)) for p in lab_paths]
    else:
        print(f"[open_ngff_pyramids] No labels found at {lab_group_path}")

    return img_levels, lab_levels, img_axes, lab_axes

# --- example usage ---
img_levels, lab_levels, img_axes, lab_axes = open_ngff_pyramids("/mnt/DATA/mouse_1.ome.zarr", label_name="seg")

print("Image pyramid shapes:", [a.shape for a in img_levels])
print("Label pyramid shapes:", [a.shape for a in lab_levels])
# Access level-0 (highest res):
img0 = img_levels[0]
lab0 = lab_levels[0] if lab_levels else None


Image pyramid shapes: [(3, 30089, 49350), (3, 6017, 9870), (3, 1203, 1974)]
Label pyramid shapes: [(30089, 49350), (6017, 9870), (1203, 1974)]


In [4]:
viewer = napari.Viewer(title = 'loading baptistes labels')
viewer.add_image(img_levels, channel_axis=0,)# scale = (2.0, 0.1625, 0.1625))

[<Image layer 'Image' at 0x7cc3f0285310>,
 <Image layer 'Image [1]' at 0x7cc418af2290>,
 <Image layer 'Image [2]' at 0x7cc3cc2d19d0>]

/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/core.py:5084: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/chunk.py:279: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/core.py:5084: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/chunk.py:279: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/core.py:5084: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/dask/array/chunk.py:279: Run

In [5]:
viewer.add_labels(lab_levels)

<Labels layer 'lab_levels' at 0x7cc350659250>

In [7]:
viewer.add_labels(lab0)
viewer.add_image(img_levels[0], channel_axis=0,)# scale = (2.0, 0.1625, 0.1625))

/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30089, 49350) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30089, 49350) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/micro-sam/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30089, 49350) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image [3]' at 0x7cbe79ccf5d0>,
 <Image layer 'Image [4]' at 0x7cc3f136b650>,
 <Image layer 'Image [5]' at 0x7cc3f02fef90>]